# Stage 3 — Liquidity & Execution Analysis

Uses validated rolling-tenor 60-min data (Butterfly + Spread) to analyze:
- MFI classification (Green/Fade/Fake/Squat) per bar
- Session liquidity ranking (Morning/Afternoon/Night)
- Slippage estimation via True Range
- Capacity / market impact proxy
- MPOB release day effects
- Intraday touch vs daily-close signal comparison
- Intraday risk re-check vs Stage 2 daily-bar figures

All results logged in stage3_minute_rebuild_log.txt.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta

PROJECT_ROOT = Path(os.getcwd()).parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'MRBackTest'))

from shared.tenor_mapping import front_month, tenor_to_contract_month, contract_month_to_str

OUTPUT_DIR = Path('.')

# Load rebuilt tenor data
bfly_long = pd.read_csv('butterfly_tenor_full_long.csv', parse_dates=['datetime'])
spd_long = pd.read_csv('spread_tenor_full_long.csv', parse_dates=['datetime'])
bfly_long = bfly_long[bfly_long['datetime'] >= '2024-01-01']
spd_long = spd_long[spd_long['datetime'] >= '2024-01-01']

INSTRUMENT_MAP = {
    'M1-M2': ('spread', 'Current'), 'M2-M3': ('spread', '+1M'),
    'M3-M4': ('spread', '+2M'), 'M4-M5': ('spread', '+3M'),
    'M5-M6': ('spread', '+4M'),
    'BF_M1M2M3': ('butterfly', 'Current'), 'BF_M2M3M4': ('butterfly', '+1M'),
    'BF_M3M4M5': ('butterfly', '+2M'), 'BF_M4M5M6': ('butterfly', '+3M'),
}

instruments = {}
for inst_name, (inst_type, tenor) in INSTRUMENT_MAP.items():
    src = spd_long if inst_type == 'spread' else bfly_long
    data = src[src['tenor'] == tenor].copy().sort_values('datetime').reset_index(drop=True)
    instruments[inst_name] = data
    print(f'{inst_name}: {len(data)} bars')

## Part 1 — MFI Classification

In [ ]:
for inst_name, df in instruments.items():
    df['total_vol'] = df['avol'].fillna(0) + df['bvol'].fillna(0)
    df['mfi_prev'] = df['mfi'].shift(1)
    df['vol_prev'] = df['total_vol'].shift(1)
    mfi_up = df['mfi'] > df['mfi_prev']
    vol_up = df['total_vol'] > df['vol_prev']
    cls = pd.Series('', index=df.index, dtype='object')
    cls[mfi_up & vol_up] = 'Green'
    cls[~mfi_up & ~vol_up] = 'Fade'
    cls[mfi_up & ~vol_up] = 'Fake'
    cls[~mfi_up & vol_up] = 'Squat'
    cls.iloc[0] = ''
    cls[df['mfi'].isna() | df['mfi_prev'].isna()] = ''
    df['mfi_class'] = cls
    instruments[inst_name] = df

print('Classification distribution:')
for inst_name, df in instruments.items():
    classified = df[df['mfi_class'] != '']
    n = len(classified)
    pcts = classified['mfi_class'].value_counts(normalize=True) * 100
    print(f'  {inst_name:<12} G={pcts.get("Green",0):.1f}% F={pcts.get("Fade",0):.1f}% '
          f'Fk={pcts.get("Fake",0):.1f}% Sq={pcts.get("Squat",0):.1f}% (n={n})')

## Part 2 — Session Ranking

Sessions in MYT (UTC+8): Morning 10:30-12:30, Afternoon 14:30-17:30, Night 21:30-23:30

In [ ]:
def assign_session(dt_utc):
    myt_hour = (dt_utc.hour + 8) % 24
    myt_min = dt_utc.minute
    myt_time = myt_hour * 100 + myt_min
    if 1030 <= myt_time <= 1230: return 'Morning'
    elif 1430 <= myt_time <= 1730: return 'Afternoon'
    elif 2130 <= myt_time <= 2330: return 'Night'
    return 'Other'

for inst_name, df in instruments.items():
    df['session'] = df['datetime'].apply(assign_session)

# Composite score per session per instrument
print('Composite Score = (%Green + %Squat) - %Fake')
print(f'{"Instrument":<12} {"Morning":>9} {"Afternoon":>11} {"Night":>9}')
print('-'*45)
for inst_name, df in instruments.items():
    scores = []
    for session in ['Morning', 'Afternoon', 'Night']:
        sdf = df[(df['session'] == session) & (df['mfi_class'] != '')]
        if len(sdf) < 10:
            scores.append(np.nan)
            continue
        pcts = sdf['mfi_class'].value_counts(normalize=True) * 100
        score = (pcts.get('Green', 0) + pcts.get('Squat', 0)) - pcts.get('Fake', 0)
        scores.append(score)
    print(f'{inst_name:<12} {scores[0]:>8.1f}% {scores[1]:>10.1f}% {scores[2]:>8.1f}%')

## Parts 3-7

See `stage3_minute_rebuild_log.txt` for full results of:
- Part 3: Slippage estimation (True Range by session)
- Part 4: Capacity / market impact proxy
- Part 5: MPOB release day comparison
- Part 6: Intraday touch vs daily-close signal comparison
- Part 7: Intraday risk re-check vs Stage 2 daily-bar figures